tuned_metric_path = f"Model_Results_TUNED_{CITY}.csv"
untuned_metric_path = f"Model_Results_TUNED_{CITY}.csv"

In [61]:
import pandas as pd
pd.set_option('display.max_columns', None)


In [62]:
cities = [
    'ahmedabad', 'delhi', 'pune', 'kanpur', 'mumbai',
    'srinagar', 'jaipur', 'lucknow', 'bangalore', 'patna', 'varanasi', 'amritsar',
    'bhopal', 'chandigarh', 'chennai', 'dehradun', 'hyderabad',
    'indore', 'kolkata', 'ludhiana', 'surat', 'thiruvananthapuram', 'visakhapatnam'
]

def prepare_dataframe(MODE):
    tuned_dfs = []
    for city in cities:
        tuned_metric_path = f"results/Model_Results_{MODE}_{city}.csv"
        # r2_col = 'R²'
        # r2_col = 'RMSE'
        r2_col = 'MAE'
        try:
            tuned_df = pd.read_csv(tuned_metric_path)
            tuned_df = tuned_df[['Model', r2_col]].copy()
            tuned_df['City'] = city
            tuned_dfs.append(tuned_df)
        except FileNotFoundError:
            print(f"Warning: Tuned file not found for city {city}: {tuned_metric_path}")
        except Exception as e:
            print(f"Error processing {tuned_metric_path}: {e}")
    tuned_combined = pd.concat(tuned_dfs, ignore_index=True)
    r2_pivot = tuned_combined.pivot(index='City', columns='Model', values=tuned_combined.columns[1])
    r2_pivot = r2_pivot.sort_index().sort_index(axis=1)
    return r2_pivot
    

In [63]:
tuned_df = prepare_dataframe(MODE='TUNED')
untuned_df = prepare_dataframe(MODE='UNTUNED')
print(tuned_df.shape, untuned_df.shape)

(23, 12) (23, 12)


In [64]:
tuned_df

Model,CatBoost,Decision Tree,ElasticNet,ExtraTreeRegressor,HistGradientBoosting,KNN,Lasso Regression,LightGBM,Random Forest,Ridge Regression,SVR,XGBoost
City,,,,,,,,,,,,
ahmedabad,8.1975,11.5572,82.8952,8.2971,8.1396,9.2004,8.7470,7.5470,7.3057,46.8990,10.2927,7.1616
amritsar,9.8133,10.8509,52.9908,9.9405,10.2872,12.4178,52.9908,9.4507,10.9588,66.8898,14.7923,8.8904
bangalore,6.3770,4.5754,4.3985,4.9767,4.9251,6.7628,4.3371,5.2625,4.5296,14.4054,10.6859,5.2105
bhopal,15.4751,19.8953,68.1078,14.3125,15.6190,17.1850,49.7705,16.0019,14.4513,76.9841,21.7006,14.4761
chandigarh,25.2147,25.8071,60.8830,22.5102,24.0138,33.0264,61.2740,28.0516,26.4531,61.3951,35.9894,23.9643
chennai,5.9684,8.7871,29.8809,8.6766,6.2940,11.6524,9.6404,4.8379,6.1783,21.3095,9.7613,5.5886
dehradun,10.9138,16.3887,17.1191,13.9550,31.9889,28.9512,17.1191,17.3265,10.6357,70.6002,32.2725,16.2333
delhi,19.4649,35.8389,1310.6240,17.2407,25.5016,50.0170,1310.6240,31.7950,29.1345,1310.5399,50.5700,21.9771
hyderabad,3.0998,5.4352,259.5623,6.2772,3.2746,11.1326,259.5623,4.1934,5.1593,177.9685,8.1769,3.3879


In [65]:
untuned_df

Model,CatBoost,Decision Tree,ElasticNet,ExtraTreeRegressor,HistGradientBoosting,KNN,Lasso Regression,LightGBM,Random Forest,Ridge Regression,SVR,XGBoost
City,,,,,,,,,,,,
ahmedabad,7.4307,9.6822,8.4843,8.1687,8.3724,8.2443,8.7463,8.8133,7.4396,113.2202,13.2151,8.5736
amritsar,10.1429,13.9959,57.9685,11.2192,10.4205,18.5141,16.6872,10.6509,10.7436,66.8898,17.8850,9.1727
bangalore,5.6251,5.9140,4.4388,6.2191,4.9337,6.7529,4.7398,5.1246,4.7616,8.4402,9.7884,5.7531
bhopal,15.6734,15.5400,39.0311,15.5998,15.6280,17.3388,21.9271,15.6930,15.0111,76.9841,20.8025,14.6477
chandigarh,26.6358,19.9342,59.0691,24.9163,24.1764,32.8825,34.1640,24.3067,25.8011,61.3951,34.0100,26.3859
chennai,6.1238,6.7481,8.3773,7.1720,5.2764,10.6227,9.1357,4.4117,5.6642,18.4030,10.2417,3.8004
dehradun,13.2274,20.3936,87.1476,39.8932,24.8988,31.2576,17.1191,21.2367,11.0441,95.9045,21.4220,10.2841
delhi,23.7893,45.0456,1158.8076,23.7955,26.1861,50.0007,1174.6992,23.2721,29.1389,1311.9041,49.4576,31.9131
hyderabad,3.4350,6.6027,268.5068,6.2772,3.4169,13.0319,5.5988,3.7627,5.0953,212.6668,7.8874,4.4166


In [66]:
# For each city and model, if the untuned R² is greater than the tuned R², update tuned_df with the untuned value.
# This ensures that tuned_df always reflects the better (higher) R² value for each (city, model) pair.

# Iterate over the intersection of indices and columns to ensure alignment
common_cities = tuned_df.index.intersection(tuned_df.index)
common_models = tuned_df.columns.intersection(untuned_df.columns)

for city in common_cities:
    for model in common_models:
        tuned_val = tuned_df.loc[city, model]
        untuned_val = untuned_df.loc[city, model]
        # Only update if untuned is not null and is greater than tuned
        if pd.notnull(untuned_val) and (pd.isnull(tuned_val) or untuned_val < tuned_val):
            tuned_df.loc[city, model] = untuned_val
tuned_df

Model,CatBoost,Decision Tree,ElasticNet,ExtraTreeRegressor,HistGradientBoosting,KNN,Lasso Regression,LightGBM,Random Forest,Ridge Regression,SVR,XGBoost
City,,,,,,,,,,,,
ahmedabad,7.4307,9.6822,8.4843,8.1687,8.1396,8.2443,8.7463,7.5470,7.3057,46.8990,10.2927,7.1616
amritsar,9.8133,10.8509,52.9908,9.9405,10.2872,12.4178,16.6872,9.4507,10.7436,66.8898,14.7923,8.8904
bangalore,5.6251,4.5754,4.3985,4.9767,4.9251,6.7529,4.3371,5.1246,4.5296,8.4402,9.7884,5.2105
bhopal,15.4751,15.5400,39.0311,14.3125,15.6190,17.1850,21.9271,15.6930,14.4513,76.9841,20.8025,14.4761
chandigarh,25.2147,19.9342,59.0691,22.5102,24.0138,32.8825,34.1640,24.3067,25.8011,61.3951,34.0100,23.9643
chennai,5.9684,6.7481,8.3773,7.1720,5.2764,10.6227,9.1357,4.4117,5.6642,18.4030,9.7613,3.8004
dehradun,10.9138,16.3887,17.1191,13.9550,24.8988,28.9512,17.1191,17.3265,10.6357,70.6002,21.4220,10.2841
delhi,19.4649,35.8389,1158.8076,17.2407,25.5016,50.0007,1174.6992,23.2721,29.1345,1310.5399,49.4576,21.9771
hyderabad,3.0998,5.4352,259.5623,6.2772,3.2746,11.1326,5.5988,3.7627,5.0953,177.9685,7.8874,3.3879


In [67]:
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [68]:
def plot_model_metric_heatmap(df, MODE="TUNED", Metric="R²"):
    """
    Plots a heatmap of the specified metric (e.g., R², RMSE, MAE) for each city and model.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with cities as index and models as columns, containing the metric values.
    MODE : str, optional
        A string indicating the mode or context (e.g., 'TUNED', 'UNTUNED'), used in the plot title and filename.
    Metric : str, optional
        The metric to display (e.g., 'R²', 'RMSE', 'MAE'), used in the plot title and filename.

    Returns
    -------
    None
        Saves the heatmap as a PNG file in the 'plots' directory.
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    import os

    os.makedirs('plots', exist_ok=True)

    # Create a copy of the dataframe to avoid modifying the original
    df_plot = df.copy()
    
    # Determine clipping and color range based on metric
    if Metric=="R²":
        # Clip R² values to [-1, 1] range
        df_plot = np.clip(df_plot, -1, 1)
        vmin, vmax = -1, 1
        cmap = "YlGnBu"
    elif Metric=="RMSE":
        # For RMSE, clip to reasonable range (0 to 95th percentile)
        upper_limit = 50 #np.percentile(df_plot.values, 75)
        df_plot = np.clip(df_plot, 0, upper_limit)
        vmin, vmax = 0, upper_limit
        cmap = "YlOrRd"  # Reversed colormap (lower RMSE = better = darker)
    elif Metric=="MAE":
        # For MAE, clip to reasonable range (0 to 95th percentile)
        upper_limit = 50 #np.percentile(df_plot.values, 75)
        df_plot = np.clip(df_plot, 0, upper_limit)
        vmin, vmax = 0, upper_limit
        cmap = "YlOrRd"  # Reversed colormap (lower MAE = better = darker)
    else:
        # For other metrics, use original logic
        vmin, vmax = None, None
        cmap = "YlOrRd"

    plt.figure(figsize=(12, 12))
    sns.set(font_scale=1.1)

    ax = sns.heatmap(
        df_plot,
        annot=True,
        fmt=".2f",
        cmap=cmap,
        linewidths=0.5,
        cbar=True,
        vmin=vmin, vmax=vmax,
        annot_kws={"size": 11, "weight": "bold"},
        square=False
    )
    plt.title(f"{Metric} Score by City and Model ({MODE})", fontsize=16, fontweight='bold', pad=15)
    plt.ylabel("City", fontsize=13)
    plt.xlabel("Model", fontsize=13)
    plt.yticks(rotation=0)
    plt.tight_layout()
    filename = f"plots/all_models_{Metric.lower().replace('²','2').replace(' ','_')}_table_{MODE.lower()}.png"
    plt.savefig(filename, dpi=600)
    plt.close()

In [69]:
# The following ordering groups models by their underlying algorithmic family for clarity and interpretability:
# - Tree-based ensemble methods
# - Single decision trees
# - Linear models (including regularized variants)
# - Instance-based (KNN)
# - Kernel-based (SVR)
# - Gradient boosting (including specialized libraries)

ordered_cols = [
    # Tree-based ensemble methods
    # Bagging-based ensemble methods
    'Random Forest', 'ExtraTreeRegressor',
    # Boosting-based ensemble methods
    'HistGradientBoosting', 'LightGBM', 'XGBoost', 'CatBoost',
    # Single decision tree
    'Decision Tree',
    # Linear models
    'Ridge Regression', 'Lasso Regression', 'ElasticNet',
    # Instance-based
    'KNN',
    # Kernel-based
    'SVR'
]

In [70]:
tuned_df.index = [str(city).lower() for city in tuned_df.index]
tuned_df_rounded = tuned_df.round(2)
tuned_df_rounded.to_csv(f"plots/r2.csv", index=False)
tuned_df_rounded = tuned_df_rounded[ordered_cols]

untuned_df.index = [str(city).lower() for city in untuned_df.index]
untuned_df_rounded = untuned_df.round(2)
untuned_df_rounded.to_csv(f"plots/r2_untuned.csv", index=False)
untuned_df_rounded = untuned_df_rounded[ordered_cols]


In [71]:
# plot_model_metric_heatmap(tuned_df_rounded, 'TUNED', Metric='RMSE')
# plot_model_metric_heatmap(untuned_df_rounded, 'UNTUNED', Metric='RMSE')
plot_model_metric_heatmap(tuned_df_rounded, 'TUNED', Metric='MAE')
plot_model_metric_heatmap(untuned_df_rounded, 'UNTUNED', Metric='MAE')